In [0]:
from pyspark.sql import functions as F

SILVER_POS_PATH = "/Volumes/mini_project_team_a3t7/default/mini_project/silver/pos_transactions_stream/"
CHECKPOINT_GOLD = "/Volumes/mini_project_team_a3t7/default/mini_project/checkpoint/gold/fact_sales/"

df_silver = spark.readStream.format("delta").load(SILVER_POS_PATH)

fact_sales = (
    df_silver
    .withColumn("sales_date", F.to_date("event_timestamp"))
    .withColumn("sales_hour", F.hour("event_timestamp"))
    .withColumn("_gold_processed_at", F.current_timestamp())
)

query_gold = (
    fact_sales.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", CHECKPOINT_GOLD)
        .partitionBy("sales_date")
        .trigger(availableNow=True)
        .toTable("mini_project_team_a3t7.gold.POS_RealTime")
)

query_gold.awaitTermination()